# Employee Attrition Prediction - Explainable AI & Advanced Features

This notebook covers explainability, risk scoring, timeline prediction, and recommendations.

## 1. Import Libraries and Load Models

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')
from risk_scoring import RiskScorer
from timeline_prediction import TimelinePredictor
from recommendation_engine import RecommendationEngine
import warnings
warnings.filterwarnings('ignore')

# Load test data
X_test = pd.read_csv('../data/X_test.csv')
y_test = pd.read_csv('../data/y_test.csv').squeeze()

# Load models
with open('../models/random_forest.pkl', 'rb') as f:
    rf_model = pickle.load(f)

print("Data and models loaded successfully!")

## 2. Feature Importance Analysis

In [ ]:
# Extract feature importance
feature_importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance_df.head(10))

# Visualize
plt.figure(figsize=(12, 6))
top_10 = feature_importance_df.head(10)
plt.barh(range(len(top_10)), top_10['Importance'], color='steelblue')
plt.yticks(range(len(top_10)), top_10['Feature'])
plt.xlabel('Importance Score')
plt.title('Top 10 Features Contributing to Attrition Prediction')
plt.tight_layout()
plt.savefig('../explainability/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Attrition Risk Scoring (0-100)

In [ ]:
# Calculate risk scores for test set
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
risk_scores = y_pred_proba * 100

# Add to test data
X_test_with_risk = X_test.copy()
X_test_with_risk['Risk_Score'] = risk_scores
X_test_with_risk['Actual_Attrition'] = y_test.values

# Risk categories
def get_risk_category(score):
    if score < 25:
        return 'Low Risk'
    elif score < 50:
        return 'Medium Risk'
    elif score < 75:
        return 'High Risk'
    else:
        return 'Critical Risk'

X_test_with_risk['Risk_Category'] = X_test_with_risk['Risk_Score'].apply(get_risk_category)

print("\nRisk Score Distribution:")
print(X_test_with_risk[['Risk_Score', 'Risk_Category', 'Actual_Attrition']].describe())

print("\nRisk Category Distribution:")
print(X_test_with_risk['Risk_Category'].value_counts())

## 4. Visualize Risk Scores

In [ ]:
# Plot risk score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(X_test_with_risk['Risk_Score'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Risk Score (0-100)')
axes[0].set_ylabel('Number of Employees')
axes[0].set_title('Distribution of Attrition Risk Scores')
axes[0].axvline(25, color='green', linestyle='--', label='Low-Medium threshold')
axes[0].axvline(50, color='orange', linestyle='--', label='Medium-High threshold')
axes[0].axvline(75, color='red', linestyle='--', label='High-Critical threshold')
axes[0].legend()

# Box plot by actual attrition
X_test_with_risk.boxplot(column='Risk_Score', by='Actual_Attrition', ax=axes[1])
axes[1].set_xlabel('Actual Attrition')
axes[1].set_ylabel('Risk Score')
axes[1].set_title('Risk Scores by Actual Attrition Status')
axes[1].get_figure().suptitle('')

plt.tight_layout()
plt.savefig('../explainability/risk_scores_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Timeline Prediction

In [ ]:
# Timeline prediction based on risk score
def predict_timeline(risk_score):
    if risk_score < 30:
        return '>12 months'
    elif risk_score < 50:
        return '6-12 months'
    elif risk_score < 70:
        return '3-6 months'
    else:
        return '0-3 months'

X_test_with_risk['Predicted_Timeline'] = X_test_with_risk['Risk_Score'].apply(predict_timeline)

print("\nTimeline Distribution:")
print(X_test_with_risk['Predicted_Timeline'].value_counts())

# Sample high-risk employees
high_risk = X_test_with_risk[X_test_with_risk['Risk_Score'] >= 75].head(10)
print("\nSample High-Risk Employees:")
print(high_risk[['Risk_Score', 'Risk_Category', 'Predicted_Timeline', 'Actual_Attrition']].to_string())

## 6. HR Recommendation Engine

In [ ]:
# Define recommendations based on risk category
recommendations_map = {
    'Low Risk': [
        'Regular performance monitoring',
        'Maintain current benefits',
        'Annual career development discussion'
    ],
    'Medium Risk': [
        'Schedule monthly check-in with manager',
        'Review compensation package',
        'Enroll in professional development program'
    ],
    'High Risk': [
        'Immediate manager intervention (bi-weekly meetings)',
        'Salary review and potential increase',
        'Leadership development or promotion opportunity',
        'Flexible work arrangement discussion'
    ],
    'Critical Risk': [
        'URGENT: Immediate HR + Manager meeting',
        'Retention bonus offer (10-15% increase)',
        'Fast-track promotion or role change',
        'Wellness program enrollment',
        'Work-life balance intervention (reduce overtime)'
    ]
}

# Add recommendations
X_test_with_risk['Recommendations'] = X_test_with_risk['Risk_Category'].apply(
    lambda x: ' | '.join(recommendations_map.get(x, []))
)

print("\nSample Recommendations:")
sample = X_test_with_risk[['Risk_Category', 'Recommendations']].drop_duplicates()
for _, row in sample.iterrows():
    print(f"\n{row['Risk_Category']}:")
    for rec in row['Recommendations'].split(' | '):
        print(f"  • {rec}")

## 7. Department-wise Attrition Analysis

In [ ]:
# If Department column exists in original data
# Add example analysis
dept_analysis = pd.DataFrame({
    'Risk_Score': X_test_with_risk['Risk_Score'],
    'Actual_Attrition': X_test_with_risk['Actual_Attrition']
})

print("\nOverall Statistics:")
print(f"Average Risk Score: {dept_analysis['Risk_Score'].mean():.2f}")
print(f"Employees with Risk > 50: {(dept_analysis['Risk_Score'] > 50).sum()}")
print(f"Employees with Risk > 75: {(dept_analysis['Risk_Score'] > 75).sum()}")

## 8. What-if Analysis (Salary & Overtime Adjustments)

In [ ]:
# Example: Impact of salary increase on risk
print("\nWhat-if Analysis: Impact of Interventions")
print("="*60)

# Simulated impact scenarios
scenarios = {
    'Current State': X_test_with_risk['Risk_Score'].mean(),
    'With 10% Salary Increase': X_test_with_risk['Risk_Score'].mean() * 0.85,
    'With Overtime Reduction': X_test_with_risk['Risk_Score'].mean() * 0.80,
    'With Career Development': X_test_with_risk['Risk_Score'].mean() * 0.90,
    'With All Interventions': X_test_with_risk['Risk_Score'].mean() * 0.70
}

for scenario, avg_risk in scenarios.items():
    improvement = ((X_test_with_risk['Risk_Score'].mean() - avg_risk) / X_test_with_risk['Risk_Score'].mean()) * 100
    print(f"{scenario:.<40} Avg Risk: {avg_risk:.2f} (↓{improvement:.1f}%)")

## 9. Save Comprehensive Report

In [ ]:
# Save risk and recommendation data
report_data = X_test_with_risk[['Risk_Score', 'Risk_Category', 'Predicted_Timeline', 'Recommendations', 'Actual_Attrition']].copy()
report_data.to_csv('../results/risk_assessment_report.csv', index=False)

# Summary statistics
with open('../results/explainability_summary.txt', 'w') as f:
    f.write("Employee Attrition - Explainability & Risk Assessment Report\n")
    f.write("="*70 + "\n\n")
    
    f.write("Key Findings:\n")
    f.write(f"  - Total Employees Analyzed: {len(X_test_with_risk)}\n")
    f.write(f"  - Average Risk Score: {X_test_with_risk['Risk_Score'].mean():.2f}\n")
    f.write(f"  - High Risk (Score > 75): {(X_test_with_risk['Risk_Score'] > 75).sum()} employees\n")
    f.write(f"  - Medium Risk (50-75): {((X_test_with_risk['Risk_Score'] >= 50) & (X_test_with_risk['Risk_Score'] <= 75)).sum()} employees\n\n")
    
    f.write("Top 5 Critical Risk Features:\n")
    for idx, row in feature_importance_df.head(5).iterrows():
        f.write(f"  {row['Feature']}: {row['Importance']:.4f}\n")

print("\nReports saved successfully!")
print("  - risk_assessment_report.csv")
print("  - explainability_summary.txt")